# Level 1 — Pandas Fundamentals

**Dataset:** Population of Tanzanian regions, 2022 Population and Housing Census (National Bureau of Statistics — nbs.go.tz)

This is a subset of confirmed regions from the census (not all 31 — some figures aren't published at the granularity needed here). `Area_km2` is intentionally missing for a few regions, so we get real practice handling `NaN` values rather than working with an artificially clean dataset.

**What you'll practice:**
- Building a DataFrame from scratch
- Inspecting data (`.head()`, `.info()`, `.describe()`)
- Sorting and filtering
- `groupby()` aggregation
- Handling missing values
- Basic derived columns (population density)

In [2]:
import pandas as pd
import numpy as np

print("pandas version:", pd.__version__)

pandas version: 2.3.3


## 1. Build the dataset

In [3]:
data = {
    "Region": [
        "Dar es Salaam", "Mwanza", "Tabora", "Morogoro", "Dodoma",
        "Arusha", "Geita", "Songwe", "Iringa", "Mjini Magharibi", "Njombe"
    ],
    "Zone": [
        "Coastal", "Lake", "Western", "Eastern", "Central",
        "Northern", "Lake", "Southern Highlands", "Southern Highlands", "Zanzibar", "Southern Highlands"
    ],
    "Population_2022": [
        5383728, 3699872, 3391679, 3197104, 3085625,
        2356255, 2977608, 1344687, 1192728, 893169, 889946
    ],
    "Area_km2": [
        1393, np.nan, 76151, np.nan, 41311,
        37576, 20054, 27656, 35503, 230, np.nan
    ]
}

df = pd.DataFrame(data)
df

,Region,Zone,Population_2022,Area_km2
0,Dar es Salaam,Coastal,5383728,1393.0
1,Mwanza,Lake,3699872,NaN
2,Tabora,Western,3391679,76151.0
3,Morogoro,Eastern,3197104,NaN
4,Dodoma,Central,3085625,41311.0
5,Arusha,Northern,2356255,37576.0
6,Geita,Lake,2977608,20054.0
7,Songwe,Southern Highlands,1344687,27656.0
8,Iringa,Southern Highlands,1192728,35503.0
9,Mjini Magharibi,Zanzibar,893169,230.0


## 2. First look at the data

In [4]:
df.head()

,Region,Zone,Population_2022,Area_km2
0,Dar es Salaam,Coastal,5383728,1393.0
1,Mwanza,Lake,3699872,NaN
2,Tabora,Western,3391679,76151.0
3,Morogoro,Eastern,3197104,NaN
4,Dodoma,Central,3085625,41311.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Region           11 non-null     object 
 1   Zone             11 non-null     object 
 2   Population_2022  11 non-null     int64  
 3   Area_km2         8 non-null      float64
dtypes: float64(1), int64(1), object(2)
memory usage: 480.0+ bytes


In [6]:
df.describe()

,Population_2022,Area_km2
count,1.100000e+01,8.000000
mean,2.582946e+06,29984.250000
std,1.406078e+06,24362.034056
min,8.899460e+05,230.000000
25%,1.268708e+06,15388.750000
50%,2.977608e+06,31579.500000
75%,3.294392e+06,38509.750000
max,5.383728e+06,76151.000000


Notice `count` for `Area_km2` is lower than `Population_2022` — that's the 3 missing values showing up. `.describe()` silently ignores NaNs when computing mean/std, which is worth knowing: your stats are only as good as the subset that has data.

## 3. Sorting and filtering

In [7]:
# Sort by population, descending
df.sort_values("Population_2022", ascending=False)

,Region,Zone,Population_2022,Area_km2
0,Dar es Salaam,Coastal,5383728,1393.0
1,Mwanza,Lake,3699872,NaN
2,Tabora,Western,3391679,76151.0
3,Morogoro,Eastern,3197104,NaN
4,Dodoma,Central,3085625,41311.0
6,Geita,Lake,2977608,20054.0
5,Arusha,Northern,2356255,37576.0
7,Songwe,Southern Highlands,1344687,27656.0
8,Iringa,Southern Highlands,1192728,35503.0
9,Mjini Magharibi,Zanzibar,893169,230.0


In [8]:
# Filter: regions with population over 3 million
df[df["Population_2022"] > 3_000_000]

,Region,Zone,Population_2022,Area_km2
0,Dar es Salaam,Coastal,5383728,1393.0
1,Mwanza,Lake,3699872,NaN
2,Tabora,Western,3391679,76151.0
3,Morogoro,Eastern,3197104,NaN
4,Dodoma,Central,3085625,41311.0


In [9]:
# Filter: Southern Highlands zone only
df[df["Zone"] == "Southern Highlands"]

,Region,Zone,Population_2022,Area_km2
7,Songwe,Southern Highlands,1344687,27656.0
8,Iringa,Southern Highlands,1192728,35503.0
10,Njombe,Southern Highlands,889946,NaN


## 4. Groupby aggregation

In [10]:
# Total population per zone
df.groupby("Zone")["Population_2022"].sum().sort_values(ascending=False)

Zone
Lake                  6677480
Coastal               5383728
Southern Highlands    3427361
Western               3391679
Eastern               3197104
Central               3085625
Northern              2356255
Zanzibar               893169
Name: Population_2022, dtype: int64

In [11]:
# Multiple aggregations at once
df.groupby("Zone")["Population_2022"].agg(["count", "sum", "mean"])

,count,sum,mean
Zone,,,
Central,1,3085625,3.085625e+06
Coastal,1,5383728,5.383728e+06
Eastern,1,3197104,3.197104e+06
Lake,2,6677480,3.338740e+06
Northern,1,2356255,2.356255e+06
Southern Highlands,3,3427361,1.142454e+06
Western,1,3391679,3.391679e+06
Zanzibar,1,893169,8.931690e+05


## 5. Handling missing values

Three regions are missing `Area_km2`. In real projects you rarely just drop rows blindly — the right move depends on what you're about to do with the data.

In [12]:
# Which rows have missing area?
df[df["Area_km2"].isna()]

,Region,Zone,Population_2022,Area_km2
1,Mwanza,Lake,3699872,NaN
3,Morogoro,Eastern,3197104,NaN
10,Njombe,Southern Highlands,889946,NaN


In [13]:
# Option A: drop rows with missing area (only makes sense if area is essential for the analysis)
df_complete = df.dropna(subset=["Area_km2"])
df_complete

,Region,Zone,Population_2022,Area_km2
0,Dar es Salaam,Coastal,5383728,1393.0
2,Tabora,Western,3391679,76151.0
4,Dodoma,Central,3085625,41311.0
5,Arusha,Northern,2356255,37576.0
6,Geita,Lake,2977608,20054.0
7,Songwe,Southern Highlands,1344687,27656.0
8,Iringa,Southern Highlands,1192728,35503.0
9,Mjini Magharibi,Zanzibar,893169,230.0


In [14]:
# Option B: keep all rows, just flag which ones are missing area
df["Area_known"] = df["Area_km2"].notna()
df[["Region", "Area_km2", "Area_known"]]

,Region,Area_km2,Area_known
0,Dar es Salaam,1393.0,True
1,Mwanza,NaN,False
2,Tabora,76151.0,True
3,Morogoro,NaN,False
4,Dodoma,41311.0,True
5,Arusha,37576.0,True
6,Geita,20054.0,True
7,Songwe,27656.0,True
8,Iringa,35503.0,True
9,Mjini Magharibi,230.0,True


## 6. Derived column: population density

Only calculable where we have area data — this naturally produces `NaN` for the 3 regions missing area, which is the *correct* behavior (better than guessing).

In [15]:
df["Density_per_km2"] = (df["Population_2022"] / df["Area_km2"]).round(1)
df[["Region", "Population_2022", "Area_km2", "Density_per_km2"]].sort_values("Density_per_km2", ascending=False)

,Region,Population_2022,Area_km2,Density_per_km2
9,Mjini Magharibi,893169,230.0,3883.3
0,Dar es Salaam,5383728,1393.0,3864.8
6,Geita,2977608,20054.0,148.5
4,Dodoma,3085625,41311.0,74.7
5,Arusha,2356255,37576.0,62.7
7,Songwe,1344687,27656.0,48.6
2,Tabora,3391679,76151.0,44.5
8,Iringa,1192728,35503.0,33.6
1,Mwanza,3699872,NaN,NaN
3,Morogoro,3197104,NaN,NaN


## Exercises

Try these yourself before moving to Level 2:

1. Find the region with the smallest population in this dataset.
2. Calculate what percentage of the total population (across all rows) lives in the Lake zone.
3. Add a new column `Population_millions` that expresses `Population_2022` in millions, rounded to 2 decimal places.
4. Using `.loc[]`, select only the `Region` and `Density_per_km2` columns for regions where density is above 1000/km².
5. (Challenge) Write a function that takes a zone name and returns the region with the highest population in that zone, then apply it to each zone.

In [16]:
df.loc[df["Population_2022"].idxmin(), "Region"]

'Njombe'

In [17]:
# Your exercise answers here
lake_population = df[df["Zone"] == "Lake"]["Population_2022"].sum()
total_population = df["Population_2022"].sum()

percentage = (lake_population / total_population) * 100
round(percentage, 1)

np.float64(23.5)

In [18]:
df["Population_millions"] = (df["Population_2022"] / 1_000_000).round(2)
df[["Region", "Population_2022", "Population_millions"]]


,Region,Population_2022,Population_millions
0,Dar es Salaam,5383728,5.38
1,Mwanza,3699872,3.70
2,Tabora,3391679,3.39
3,Morogoro,3197104,3.20
4,Dodoma,3085625,3.09
5,Arusha,2356255,2.36
6,Geita,2977608,2.98
7,Songwe,1344687,1.34
8,Iringa,1192728,1.19
9,Mjini Magharibi,893169,0.89


In [19]:
df.loc[df["Density_per_km2"] > 1000, ["Region", "Density_per_km2"]]

,Region,Density_per_km2
0,Dar es Salaam,3864.8
9,Mjini Magharibi,3883.3


In [20]:
df.loc[df.groupby("Zone")["Population_2022"].idxmax(), ["Zone", "Region"]]

,Zone,Region
4,Central,Dodoma
0,Coastal,Dar es Salaam
3,Eastern,Morogoro
1,Lake,Mwanza
5,Northern,Arusha
7,Southern Highlands,Songwe
2,Western,Tabora
9,Zanzibar,Mjini Magharibi


In [21]:
def top_region_in_zone(zone_name):
    zone_df = df[df["Zone"] == zone_name]
    top_row = zone_df.loc[zone_df["Population_2022"].idxmax()]
    return top_row["Region"]

for zone in df["Zone"].unique():
    print(zone, "->", top_region_in_zone(zone))

Coastal -> Dar es Salaam
Lake -> Mwanza
Western -> Tabora
Eastern -> Morogoro
Central -> Dodoma
Northern -> Arusha
Southern Highlands -> Songwe
Zanzibar -> Mjini Magharibi
